In [1]:
import pickle

In [2]:
with open("egypt_digital.pkl", "rb") as f:
    train = pickle.load(f)

In [3]:
train

[Document(metadata={'source': 'https://digital.gov.eg/categories/terms/استمارة-تحديث-بيانات-المواطن'}, page_content='مصر الرقمية تصفح الخدمات دليل الجهات الحكومية تسجيل الدخول المزيد كل الخدمات الرقمية خدمات الكارت الموحد استمارة تحديث بيانات المواطن استمارة تحديث بيانات المواطن وصف الخدمة تتيح هذه الخدمة للمستخدمين تحديث بياناتهم الشخصية لدى الجهة المختصة، وذلك لضمان دقة المعلومات المسجلة وتحسين جودة الخدمات المقدمة شروط و أحكام الخدمة  التأكد من صحة ودقة البيانات المقدمة طبقا لبطاقة الرقم القومي  الالتزام بتقديم بيانات محدثة وخالية من أي معلومات مضللة أو خاطئة  استخدام الخدمة فقط للأغراض القانونية والشخصية  التحقق من صحة البيانات المقدمة قبل اعتماد التحديث  رفض أي طلب يحتوي على بيانات غير مكتملة أو مزيفة  لا تتحمل الجهة أي مسؤولية عن أي أضرار ناتجة عن إدخال بيانات غير صحيحة من قبل المستخدم  في حال اكتشاف أي تلاعب أو محاولة انتحال هوية، سيتم اتخاذ الإجراءات القانونية اللازمة  تُعد البيانات المقدمة من خلال هذه الخدمة ملزمة قانونيًا، ويترتب عليها تحديث السجلات الرسمية صاحبة الولاية  في 

## Build the Chroma vector store

Embeds the documents loaded above and persists them to `infloat/`, using the same embedding model and persist directory the app reads at runtime via `config.vector_store_config` (see `bootstrap/vectorstore_provider.py`). Run this cell whenever `egypt_digital.pkl` changes, to rebuild the index the app queries.

In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

from config import vector_store_config as config

config

In [ ]:
embedding = HuggingFaceEmbeddings(model_name=config.embeddings_name)

vector_store = Chroma.from_documents(
    documents=train,
    embedding=embedding,
    persist_directory=config.persist_directory,
)

print(f"Persisted {vector_store._collection.count()} vectors to '{config.persist_directory}/'")

### Sanity check

Confirms the persisted store actually retrieves relevant documents before trusting it in the app.

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": config.top_k})
results = retriever.invoke("كيف أستخرج بطاقة الرقم القومي؟")

for doc in results:
    print(doc.metadata.get("source"))
    print(doc.page_content[:200])
    print("---")